In [1]:
source(here::here('MG3.functions1.R'))
source(here::here('MG3.functions9.R'))
library(latex2exp)
library(fpc)

# Running analysis for each HMMid

In [2]:
queryTxt='SELECT DISTINCT ensembl_gene_id FROM AllSpeciesFoundGeneNames5MB'
HMMs=dbGetQuery(mydbRO, queryTxt)[[1]]

In [3]:
scriptsDir=normalizePath(path.expand(paste0(OutputDir,'../notebooks/')))
RscriptPath=paste0(scriptsDir, "/MG3.Step8.runningAnHMM.5MB.R")
RscriptCommands=paste(RscriptPath, HMMs)

In [4]:
# create dir
comToBash=paste0("mkdir -p ",OutputDir,"step8.HMMgenomeDescriptions/slurm_logs")
system(comToBash, intern = FALSE, wait = TRUE)
# saving the commands
writeLines(RscriptCommands, paste0(OutputDir,"step8.HMMgenomeDescriptions/RscriptCommands.txt"))

In [5]:
lowerLimits=1
upperLimits=NULL
maxChunk=5
maxLim=length(RscriptCommands)
while (max(lowerLimits)<maxLim){
    upperLimit=as.integer(min(max(lowerLimits)+maxChunk, maxLim))
    upperLimits=addToVec(upperLimits, upperLimit)
    lowerLimits=addToVec(lowerLimits, upperLimit+1)
}
lowerLimits=lowerLimits[ 1:(length(lowerLimits)-1)]
lowerLimits=as.integer(lowerLimits)
upperLimits=as.integer(upperLimits)

In [6]:
options(scipen = 999)
sbatchscript=paste0(normalizePath(path.expand(paste0(OutputDir,'../notebooks/'))),"/Rscript.wrapper.serial.lambdaenv.sh ",OutputDir,"/step8.HMMgenomeDescriptions/RscriptCommands.txt")
swarmlines=NULL 
for (i in 1:length(lowerLimits)){
        swarmline=paste(sbatchscript, lowerLimits[i], upperLimits[i])
        swarmlines=addToVec(swarmlines, swarmline)
}


length(swarmlines)

swarmparams=paste0("#SWARM -t 1 -g 40 --time=08:00:00 --gres=lscratch:100 --logdir ", path.expand(OutputDir),"step8.HMMgenomeDescriptions/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"step8.HMMgenomeDescriptions/runningEachHMMidAnalysis.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    message("Run from bash: ", fname)
    swarmFiles=addToVec(swarmFiles, fname)
    processed=upperlimit
    i=i+1
}

[1] 1601

1 1000

Run from bash: ~/data/lambda/output/step8.HMMgenomeDescriptions/runningEachHMMidAnalysis.0-2026-02-02_14.01.32.275582.swarm

1001 1601

Run from bash: ~/data/lambda/output/step8.HMMgenomeDescriptions/runningEachHMMidAnalysis.1-2026-02-02_14.01.32.275582.swarm



## Saving the results to SQL

In [10]:
#run ONCE
fs=list.files(path = paste0(OutputDir,"step8.HMMgenomeDescriptions"), pattern = "E.*RDS$", full.names = TRUE)

if ("HMMstats5MB" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE HMMstats5MB')   
i=1
for (f in fs){
    if (i %% 200 == 0) message(i," / ", length(fs))
    tmpDF=readRDS(f)
    dbWriteTable(mydbRW, "HMMstats5MB", tmpDF, append=TRUE)
    i=i+1
}

[1] 0

200 / 9518

400 / 9518

600 / 9518

800 / 9518

1000 / 9518

1200 / 9518

1400 / 9518

1600 / 9518

1800 / 9518

2000 / 9518

2200 / 9518

2400 / 9518

2600 / 9518

2800 / 9518

3000 / 9518

3200 / 9518

3400 / 9518

3600 / 9518

3800 / 9518

4000 / 9518

4200 / 9518

4400 / 9518

4600 / 9518

4800 / 9518

5000 / 9518

5200 / 9518

5400 / 9518

5600 / 9518

5800 / 9518

6000 / 9518

6200 / 9518

6400 / 9518

6600 / 9518

6800 / 9518

7000 / 9518

7200 / 9518

7400 / 9518

7600 / 9518

7800 / 9518

8000 / 9518

8200 / 9518

8400 / 9518

8600 / 9518

8800 / 9518

9000 / 9518

9200 / 9518

9400 / 9518



In [11]:
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS HMMstats5MBhmmInd ON HMMstats5MB(ensembl_gene_id, accession)')   
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS HMMstats5MBAccInd ON HMMstats5MB(accession, ensembl_gene_id)')   

[1] 0

[1] 0

# Stats over families

In [23]:
analyzeTheFamilies<-function(inDF, incolX=(inDF$DSlen)/median(inDF$DSlen),
                             incolY=2*(inDF$asRMSKlen-inDF$sRMSKlen)/inDF$DSlen, colYthreshold=0.75, Ymask=inDF$DSlen<(4000), epsilon=0.2,
                            minOddsRatio=2, minMedianDSforTopCluster=25000, minMedian_asRMSKlen10K=4000, MinPts = 10, minGenomesInTopCluster=5, verbose=FALSE){
    incolY[ Ymask] =0
    inDF$colX=incolX
    inDF$colY=incolY
    inDF=inDF[ !(is.na(incolX) | is.na(incolY)) ,]

    Dbscan_cl=dbscan(inDF[, c("colX", "colY")], eps = epsilon, MinPts = MinPts) 
    inDF$cluster=Dbscan_cl$cluster
    clustersNormDF=inDF %>% group_by(cluster) %>% summarize(numGenomes=n(), numFamilies=length(unique(familyID)), meanColY=mean(colY),
                                                           meanColX=mean(colX), meanDSlen=mean(DSlen), medianDSlen=median(DSlen), mean_asRMSKlen10K=mean(asRMSKlen10K), median_asRMSKlen10K=median(asRMSKlen10K))

    if (all(clustersNormDF$meanColY<colYthreshold)){
        if (verbose) message('all(clustersNormDF$meanColY<colYthreshold)')
        return(FALSE)
    }
    if (verbose) print(clustersNormDF, width = Inf)
    if (all(clustersNormDF$numGenomes[clustersNormDF$meanColY>=colYthreshold]<minGenomesInTopCluster)){
        if (verbose) message('all(clustersNormDF$numGenomes[clustersNormDF$meanColY>=colYthreshold]<minGenomesInTopCluster)')
        return(FALSE)
    }
    if (all(clustersNormDF$medianDSlen[ clustersNormDF$meanColY>=colYthreshold ]<minMedianDSforTopCluster)){
        if (verbose) message('all(clustersNormDF$meanDSlen<minMedianDSforTopCluster)')
        return(FALSE)
    }

    if (all(clustersNormDF$median_asRMSKlen10K[ clustersNormDF$medianDSlen>=minMedianDSforTopCluster ]<minMedian_asRMSKlen10K)){
        if (verbose) message('all(clustersNormDF$median_asRMSKlen10K[ clustersNormDF$medianDSlen>=minMedianDSforTopCluster ]<minMedian_asRMSKlen10K)')
        return(FALSE)
    }
    interestingGene=TRUE
    # 1) there should 2 or more clusters. 

    if (nrow(clustersNormDF)==1){
        if (verbose) message('nrow(clustersNormDF)==1')
        return(FALSE)
    }
    # probability that families are randomly distributed across clusters
    familyNormDF=inDF %>% group_by(familyID) %>% summarize(numGenomes=n(), weight=numGenomes/nrow(inDF), .groups="drop_last")
    familyVSclusterStat=inDF %>% group_by(familyID, cluster) %>% summarize(numGenomes=n(), .groups="drop_last")

    for (clusterID in clustersNormDF$cluster){
        familyNormDF[[paste0("cluster", clusterID,"weight")]]=0.0
    }
    
    for ( i in 1:nrow(familyNormDF)){
        famID=familyNormDF$familyID[i]
        familyTMP=inDF[ inDF$familyID==famID,]
        for (j in 1:nrow(clustersNormDF)){
            clustersNormDFtmp=clustersNormDF[ j,]
            familyClusterTMP=familyTMP[ familyTMP$cluster==clustersNormDFtmp$cluster,]
            tmpColname=paste0("cluster", clustersNormDFtmp$cluster,"weight")
            familyNormDF[i, colnames(familyNormDF)==tmpColname]=nrow(familyClusterTMP)/clustersNormDFtmp$numGenomes
        }
    }

    for ( colnum in which(grepl(pattern = "cluster\\d+weight$", colnames(familyNormDF))) ){
        familyNormDF[, ncol(familyNormDF)+1]=familyNormDF[, colnum]/familyNormDF$weight
        tmpColnames=colnames(familyNormDF)
        tmpColnames[ncol(familyNormDF)]=paste0(colnames(familyNormDF)[colnum], "ratio", collapse = "")
        colnames(familyNormDF)<-tmpColnames
    }
    #print(familyNormDF)

    # 2) At least one of the families should have more that 2x weight (odds) ratio in one of the clusters
    tmpDF=as.data.frame(familyNormDF[ ,grepl(pattern = "familyID|cluster\\d+weightratio", x = colnames(familyNormDF))]) %>% 
    rowwise() %>% filter(any(c_across(starts_with('cl'))>=minOddsRatio)) 
    
    if (nrow(tmpDF)==0){
        if (verbose) message('nrow(tmpDF)==0')
        return(FALSE)
    }
    
    overreppresentedFamiliesDF= data.frame(matrix(ncol = 2, nrow = 0))
    colnames(overreppresentedFamiliesDF) <- c("familyID", "cluster")
    #print(overreppresentedFamiliesDF)
    for (i in 1:nrow(tmpDF)){
        familyID=tmpDF$familyID[i]
        tmpDF2=tmpDF[i,grepl(pattern = "cluster\\d+weight", x = colnames(tmpDF))]
        clusterNames=colnames(tmpDF2)[tmpDF2[1,]>=minOddsRatio]
        clusterNames=str_remove_all(string = clusterNames, pattern = "[a-z]")
        for (clname in clusterNames){
            overreppresentedFamiliesDF[nrow(overreppresentedFamiliesDF)+1, ]=c(familyID, as.integer(clname))
        }
    }
    
    # 3) That family should have at least 50% of the members in that cluster
    fractionOfOverreprFamiliesInCorrCluster=NULL
    for (i in 1:nrow(overreppresentedFamiliesDF)){
        familyID=overreppresentedFamiliesDF$familyID[i]
        cluster=overreppresentedFamiliesDF$cluster[i]
        FamGenomesInCluster=familyVSclusterStat$numGenomes[ familyVSclusterStat$familyID==familyID & familyVSclusterStat$cluster==cluster]
        numGenomesInFamily=familyNormDF$numGenomes[ familyNormDF$familyID == familyID]
        fractionOfOverreprFamiliesInCorrCluster=addToVec(fractionOfOverreprFamiliesInCorrCluster, FamGenomesInCluster/numGenomesInFamily)
    }
    #print(fractionOfOverreprFamiliesInCorrCluster)
    if (!any(fractionOfOverreprFamiliesInCorrCluster>0.5)){
        if (verbose) message('!any(fractionOfOverreprFamiliesInCorrCluster>0.5)')
        return(FALSE)
    }
    return(TRUE)
}

In [24]:
queryTxt='SELECT DISTINCT ensembl_gene_id FROM HMMstats5MB'
HMMs=dbGetQuery(mydbRO, queryTxt)[[1]]
interestingGenes=NULL
Only50Kresults=NULL
i=1
for (HMMid in HMMs){
    if (i %% 200 ==0) message(i," / ",length(HMMs))
    i=i+1

    queryTxt=sprintf('SELECT * FROM HMMstats5MB WHERE ensembl_gene_id="%s"', HMMid)
    HMMstatsDT=as.data.table(dbGetQuery(mydbRO, queryTxt))
    statsDFtmp = HMMstatsDT %>% group_by(ensembl_gene_id, familyID) %>% filter(n()>2)
    normalTest=analyzeTheFamilies(statsDFtmp)
    if (normalTest){
        interestingGenes=addToVec(interestingGenes, HMMid)
        next
    }
    if (sum(statsDFtmp$DSlen > 50000)/nrow(statsDFtmp) > 0.5){
        cut50KTest=analyzeTheFamilies(statsDFtmp, incolX = as.integer(statsDFtmp$DSlen>50000), incolY=2*(statsDFtmp$asRMSKlen50K-statsDFtmp$sRMSKlen50K)/50000, 
                  epsilon = 0.1)
         if (cut50KTest){
            interestingGenes=addToVec(interestingGenes, HMMid)
             Only50Kresults=addToVec(Only50Kresults, HMMid)
        }
    }
}

200 / 9518

400 / 9518

600 / 9518

800 / 9518

1000 / 9518

1200 / 9518

1400 / 9518

1600 / 9518

1800 / 9518

2000 / 9518

2200 / 9518

2400 / 9518

2600 / 9518

2800 / 9518

3000 / 9518

3200 / 9518

3400 / 9518

3600 / 9518

3800 / 9518

4000 / 9518

4200 / 9518

4400 / 9518

4600 / 9518

4800 / 9518

5000 / 9518

5200 / 9518

5400 / 9518

5600 / 9518

5800 / 9518

6000 / 9518

6200 / 9518

6400 / 9518

6600 / 9518

6800 / 9518

7000 / 9518

7200 / 9518

7400 / 9518

7600 / 9518

7800 / 9518

8000 / 9518

8200 / 9518

8400 / 9518

8600 / 9518

8800 / 9518

9000 / 9518

9200 / 9518

9400 / 9518



In [25]:
length(interestingGenes)

[1] 42

In [26]:
length(Only50Kresults)

[1] 11

In [27]:
interestingGenes

[1] "ENSG00000001626" "ENSG00000029559" "ENSG00000081014" "ENSG00000091651"
 [5] "ENSG00000094841" "ENSG00000111701" "ENSG00000111879" "ENSG00000112167"
 [9] "ENSG00000114767" "ENSG00000115085" "ENSG00000115364" "ENSG00000115840"
[13] "ENSG00000115993" "ENSG00000116793" "ENSG00000118690" "ENSG00000121621"
[17] "ENSG00000123901" "ENSG00000124370" "ENSG00000124678" "ENSG00000130119"
[21] "ENSG00000130385" "ENSG00000133997" "ENSG00000135324" "ENSG00000135355"
[25] "ENSG00000136319" "ENSG00000139921" "ENSG00000144644" "ENSG00000144848"
[29] "ENSG00000151131" "ENSG00000151876" "ENSG00000152990" "ENSG00000163012"
[33] "ENSG00000164082" "ENSG00000166133" "ENSG00000167769" "ENSG00000170289"
[37] "ENSG00000170500" "ENSG00000177103" "ENSG00000177575" "ENSG00000183060"
[41] "ENSG00000188282" "ENSG00000203760"

In [28]:
"ENSG00000151131" %in% interestingGenes # nopchap1

[1] TRUE

In [29]:
"ENSG00000100554" %in% interestingGenes #pig ATP6V1D

[1] FALSE

In [30]:
"ENSG00000170500" %in% interestingGenes #LONRF2

[1] TRUE

In [31]:
if ("interestinggenes" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE interestinggenes')

tmpDF=data.frame(ensembl_gene_id = interestingGenes, is50Kresult= interestingGenes %in% Only50Kresults)
dbWriteTable(mydbRW, "interestinggenes", tmpDF, append=FALSE)

[1] 0